In [1]:
import sys
sys.path.append('/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject')

from functions import decompose_spectral_type, extract_number_from_spectral_type, extinction_and_error
from functions import luminosity_error_asymmetric, interpolate_value, expected_radius_error_asymmetric
from import_data import *

from astropy.constants import R_sun, L_sun, sigma_sb, G, M_sun
import re
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown as md

In [2]:
# Import data
df_hmxb = HMXB_parameters()
df_BV = pd.read_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/B_V.xlsx")
df_falenga = falenga()
df_BJ = BailerJones()
df_StellarParam =  pd.read_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/NewStellarParam/ST_colours.xlsx")

df_AllParams = pd.read_csv("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/results/ALLPARAMETERSfinal.csv")

# Objects
HMXBs = df_BV['id'].tolist()

## Some handy functions

In [3]:
# def interpolate(df2: pd.DataFrame, spectral_type: str, quantity: str, plot: bool = False):
#     """
    
#     """
#     # Make sure we don't change the input dataframe
#     df = df2.copy()
#     df = df[df['ST'].str.startswith(spectral_type[0])]
    
#     spectral_type_short, lum_class = decompose_spectral_type(spectral_type)

#     # Data for the given quantity
#     spectral_type_numbers = df['ST'].tolist()
#     spectral_type_numbers = [int(a[1]) for a in spectral_type_numbers]
#     quantity_values = df[quantity].tolist()

#     if spectral_type_short in df['ST'].tolist():
#         interpolated_value = df.loc[df['ST'] == spectral_type_short, quantity].reset_index(drop=True).at[0]
#     else:
#         # Interpolate
#         target_number = spectral_type_short[1:]
#         # print(target_number, quantity_values, spectral_type_numbers)
#         interpolated_value = float(interpolate_value(spectral_type_values=quantity_values, spectral_type_numbers=spectral_type_numbers, target_number=target_number))

#     if plot:
#         plt.plot(spectral_type_numbers, quantity_values, color='blue')
#         plt.scatter([target_number], [interpolated_value], color='orange')
#         plt.ylabel(quantity)
#         plt.xlabel('Spectral type')
#         plt.grid(True)
#         plt.show()

#     return interpolated_value

## Luminosity

In [4]:
# df = {
#     "id": [],
#     "ST": [],
#     "L": [],
#     "L_err_low": [],
#     "L_err_high": [],
#     "Teff": [],
#     # "Teff_err": [],
#     "BC": [],
#     "Mv": [],
#     "V": [],
#     "B-V": [],
#     "B-V0": [],
#     "E(B-V)": [],
#     "Av": [],
#     "Av_err": [],
#     "Rv": [],
#     "Rv_err": []
# }

# for HMXB in HMXBs:
#     ST = df_hmxb.loc[df_hmxb['id'] == HMXB, 'ST'].reset_index(drop=True).at[0]

#     V = df_BV.loc[df_BV['id'] == HMXB, 'V'].reset_index(drop=True).at[0]
#     V_err = 0
#     BV = df_BV.loc[df_BV['id'] == HMXB, 'B-V'].reset_index(drop=True).at[0]
#     BV_err = 0
#     BV0 = interpolate(df_StellarParam, ST, 'B-V')
#     BV0_err = 0

#     # Distance
#     distance = df_BJ.loc[df_BJ['id'] == HMXB, 'r_med_photogeo'].reset_index(drop=True).at[0]
#     distance_low = df_BJ.loc[df_BJ['id'] == HMXB, 'r_lo_photogeo'].reset_index(drop=True).at[0]
#     distance_high = df_BJ.loc[df_BJ['id'] == HMXB, 'r_hi_photogeo'].reset_index(drop=True).at[0]
#     Sigmad_low = distance - distance_low
#     Sigmad_high = distance_high - distance

#     BC = interpolate(df_StellarParam, ST, 'BC')
#     Teff = interpolate(df_StellarParam, ST, 'Teff')

#     # Calculate extinction
#     if 'LMC' in HMXB:
#         Rv, Rv_err = 3.40, 0
#         Av, Av_err = extinction_and_error(3.40, 0, BV, BV_err, BV0, BV0_err)
#     elif 'SMC' in HMXB:
#         Rv, Rv_err = 2.53, 0
#         Av, Av_err = extinction_and_error(2.53, 0, BV, BV_err, BV0, BV0_err)
#     else:
#         Rv, Rv_err = 3.16, 0.15
#         Av, Av_err = extinction_and_error(3.2, 0, BV, BV_err, BV0, BV0_err)
#     # Calculate E(B-V)
#     EBV = BV - BV0
#     EBV_err = np.sqrt(BV_err**2 + BV0_err**2)

#     # Calculate Absulute magnitude (visual)
#     Mv = V - 5 * np.log10(distance) + 5 - Av

#     # Calculate bolomatric absolute magnitude
#     Mbol = Mv + BC

#     # Calculate the luminosity in solar luminosities
#     L = 10**((Mbol - 4.76) / (-2.5))

#     # Calculate the error on the luminosity
#     L_err_low, L_err_high = luminosity_error_asymmetric(BC, 0, V, V_err, distance, Sigmad_high, Sigmad_low, Av, Av_err)


#     # Save
#     df['id'].append(HMXB)
#     df['ST'].append(ST)
#     df['L'].append(L)
#     df['L_err_low'].append(L_err_low)
#     df['L_err_high'].append(L_err_high)
#     df['Teff'].append(Teff)
#     # df['Teff_err'].append(Teff_err)
#     df['BC'].append(BC)
#     df['Mv'].append(Mv)
#     df['V'].append(V)
#     df['B-V'].append(BV)
#     df['B-V0'].append(BV0)
#     df['E(B-V)'].append(EBV)
#     df['Av'].append(Av)
#     df['Av_err'].append(Av_err)
#     df['Rv'].append(Rv)
#     df['Rv_err'].append(Rv_err)
#     print(Mbol)

# df = pd.DataFrame(df)

In [5]:
# df.to_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/NewStellarParam/Results.xlsx")

## Parameters with new Teff

In [18]:
for key, row in df_AllParams.iterrows():
    L, L_high, L_low = row['LBV_final'], row['LBV_final_err_high'], row['LBV_final_err_low']
    Teff = row['Teff_final']

    R, R_high, R_low = expected_radius_error_asymmetric(L, L_high, L_low, Teff, 0)

    # print(row['id'], round(R,2), round(R_high,2), round(R_low,2))
    print(f"{row['id']} \t ({round(L, 2)}) {round(np.log10(L), 2)} + {round(np.log10(L_high), 2)} - {round(np.log10(L_low), 2)} \t {Teff} \t {row['vsin(i)']}")

SMC X-1 	 (271797.71) 5.43 + 4.64 - 4.64 	 24380 	 150.0
LMC X-4 	 (97161.41) 4.99 + 4.27 - 4.27 	 32573 	 244.0
Vela X-1 	 (588169.42) 5.77 + 4.81 - 4.83 	 22370 	 110.0
Cen X-3 	 (198668.26) 5.3 + 4.65 - 4.53 	 34500 	 208.0
4U1538-52 	 (391464.53) 5.59 + 4.83 - 4.89 	 24000 	 138.0
4U1700-37 	 (463284.58) 5.67 + 4.81 - 4.73 	 35895 	 152.0
